# LIMUC supervised fine-tuning (ViT or Swin)
Fine-tune a transformer backbone for MES classification.


In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import AutoImageProcessor, AutoModelForImageClassification


2026-02-02 18:10:38.584797: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-02 18:10:38.584825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-02 18:10:38.585678: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-02 18:10:38.589842: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-02 18:10:39.494183: W tensorflow/compiler/tf2

In [2]:
# Paths & config
def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
LABEL_MAP_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "label_map.csv"
OUT_DIR = DATA_ROOT / "2_supervised_finetuning" / "out" / "finetune_vit_or_swin"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.getenv("VISION_MODEL", "google/vit-base-patch16-224-in21k")
SEED = int(os.getenv("SEED", "42"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0"))
EPOCHS = int(os.getenv("EPOCHS", "10"))
LR = float(os.getenv("LR", "3e-4"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "1e-4"))
MAX_SAMPLES = int(os.getenv("MAX_SAMPLES", "0")) or None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Output dir:", OUT_DIR)


Device: cuda
Output dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/LIMUC/2_supervised_finetuning/out/finetune_vit_or_swin


In [3]:
# Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Load metadata
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"

def to_abs(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return (images_base / p).resolve()

meta["image_path"] = meta["image_path"].apply(lambda p: str(to_abs(p)))

# Label map
if LABEL_MAP_CSV.exists():
    label_map = pd.read_csv(LABEL_MAP_CSV)
    id_to_name = dict(zip(label_map.label_id, label_map.label_name))
else:
    id_to_name = {i: name for i, name in enumerate(sorted(meta.label_name.unique()))}

name_to_id = {v: k for k, v in id_to_name.items()}
meta["label_id"] = meta["label_name"].map(name_to_id)

meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
if MAX_SAMPLES:
    meta = meta.sample(n=min(MAX_SAMPLES, len(meta)), random_state=SEED).reset_index(drop=True)

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "val"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))


Train/Val/Test: 8669 921 1686


In [4]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
size = processor.size["height"] if isinstance(processor.size, dict) else processor.size
mean = processor.image_mean
std = processor.image_std

train_tf = transforms.Compose([
    transforms.Resize(int(size * 1.15)),
    transforms.RandomResizedCrop(size, scale=(0.9, 1.0)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_tf = transforms.Compose([
    transforms.Resize(int(size * 1.15)),
    transforms.CenterCrop(size),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])


class ImageDS(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["image_path"].tolist()
        self.labels = df["label_id"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        label = int(self.labels[idx])
        return img, label


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [5]:
train_dl = DataLoader(ImageDS(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_dl = DataLoader(ImageDS(val_df, val_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_dl = DataLoader(ImageDS(test_df, val_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Class weights
class_counts = train_df["label_id"].value_counts().sort_index()
num_classes = len(class_counts)
total = class_counts.sum()
weights = total / (num_classes * class_counts)
class_weights = torch.tensor(weights.values, dtype=torch.float).to(DEVICE)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label={i: id_to_name[i] for i in range(num_classes)},
    label2id={id_to_name[i]: i for i in range(num_classes)},
).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
def run_epoch(dataloader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(is_train):
            outputs = model(pixel_values=x, labels=y)
            loss = outputs.loss
            logits = outputs.logits
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        preds = probs.argmax(axis=1)

        total_loss += loss.item() * y.size(0)
        all_probs.append(probs)
        all_preds.extend(preds)
        all_labels.extend(y.detach().cpu().numpy())

    avg_loss = total_loss / max(len(dataloader.dataset), 1)
    all_probs = np.concatenate(all_probs, axis=0) if all_probs else None
    return avg_loss, np.array(all_labels), np.array(all_preds), all_probs


In [7]:
# =====================
# Metrics helpers
# =====================
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)

try:
    from scipy.stats import spearmanr
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


def expected_calibration_error(y_true, y_prob, n_bins=10):
    if y_prob is None:
        return None
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if mask.any():
            ece += abs(accuracies[mask].mean() - confidences[mask].mean()) * mask.mean()
    return float(ece)


def compute_metrics(y_true, y_pred, labels, label_names, y_prob=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    summary = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

    if _HAS_SCIPY:
        summary["spearman"] = float(spearmanr(y_true, y_pred).correlation)

    if y_prob is not None:
        try:
            summary["auroc_ovr"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
        except Exception:
            summary["auroc_ovr"] = None
        summary["ece"] = expected_calibration_error(y_true, y_prob, n_bins=10)

    return summary, report


def save_split_outputs(
    split_name,
    y_true,
    y_pred,
    labels,
    label_names,
    out_dir,
    y_prob=None,
    df_meta=None,
):
    summary, report = compute_metrics(y_true, y_pred, labels, label_names, y_prob)

    # Save metrics
    metrics = {
        "split": split_name,
        "summary": summary,
        "report": report,
    }
    with open(out_dir / f"metrics_{split_name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Save per-class report
    per_class = {k: v for k, v in report.items() if k in label_names}
    pd.DataFrame(per_class).T.to_csv(out_dir / f"per_class_{split_name}.csv")

    # Save predictions
    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
    })
    if df_meta is not None:
        pred_df["img_id"] = df_meta["img_id"].values
        pred_df["image_path"] = df_meta["image_path"].values
    if y_prob is not None:
        for i, name in enumerate(label_names):
            pred_df[f"prob_{name}"] = y_prob[:, i]
    pred_df.to_csv(out_dir / f"pred_{split_name}.csv", index=False)

    return summary, report


In [8]:
labels = sorted(id_to_name.keys())
label_names = [id_to_name[i] for i in labels]

best_val_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, y_train, pred_train, prob_train = run_epoch(train_dl, model, optimizer)
    val_loss, y_val, pred_val, prob_val = run_epoch(val_dl, model, optimizer=None)

    val_summary, _ = compute_metrics(y_val, pred_val, labels, label_names, prob_val)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_summary["macro_f1"],
    })

    if val_summary["macro_f1"] > best_val_f1:
        best_val_f1 = val_summary["macro_f1"]
        torch.save(model.state_dict(), OUT_DIR / "best_model.pt")

    print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_macro_f1={val_summary['macro_f1']:.4f}")

# Load best
model.load_state_dict(torch.load(OUT_DIR / "best_model.pt", map_location=DEVICE))

# Final eval
_, y_train, pred_train, prob_train = run_epoch(train_dl, model, optimizer=None)
_, y_val, pred_val, prob_val = run_epoch(val_dl, model, optimizer=None)
_, y_test, pred_test, prob_test = run_epoch(test_dl, model, optimizer=None)

train_summary, _ = save_split_outputs("train", y_train, pred_train, labels, label_names, OUT_DIR, prob_train, train_df)
val_summary, _ = save_split_outputs("val", y_val, pred_val, labels, label_names, OUT_DIR, prob_val, val_df)
test_summary, _ = save_split_outputs("test", y_test, pred_test, labels, label_names, OUT_DIR, prob_test, test_df)

print("Train summary:")
print(json.dumps(train_summary, indent=2))
print("Val summary:")
print(json.dumps(val_summary, indent=2))
print("Test summary:")
print(json.dumps(test_summary, indent=2))

# Confusion matrices
val_cm = confusion_matrix(y_val, pred_val, labels=labels)
test_cm = confusion_matrix(y_test, pred_test, labels=labels)
np.save(OUT_DIR / "confusion_val.npy", val_cm)
np.save(OUT_DIR / "confusion_test.npy", test_cm)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

for split_name, cm in [("val", val_cm), ("test", test_cm)]:
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(include_values=False, cmap="Blues", ax=ax, xticks_rotation=90)
    plt.title(f"{split_name.upper()} Confusion Matrix (Transformer finetune)")
    plt.tight_layout()
    fig_path = OUT_DIR / f"confusion_{split_name}.png"
    plt.savefig(fig_path, dpi=200)
    plt.close(fig)

# Save run meta
run_meta = {
    "model": "vit_or_swin_finetune",
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "model_name": MODEL_NAME,
    "best_val_macro_f1": best_val_f1,
    "split_hash": (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").read_text().strip()
        if (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").exists() else None,
}
with open(OUT_DIR / "run_meta.json", "w") as f:
    json.dump(run_meta, f, indent=2)

pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)

print("Saved outputs to", OUT_DIR)


Epoch 1: train_loss=0.9089 val_loss=0.8585 val_macro_f1=0.4912
Epoch 2: train_loss=0.7496 val_loss=0.7270 val_macro_f1=0.6050
Epoch 3: train_loss=0.7817 val_loss=0.6728 val_macro_f1=0.6362
Epoch 4: train_loss=0.6621 val_loss=0.6083 val_macro_f1=0.6333
Epoch 5: train_loss=0.6288 val_loss=0.6743 val_macro_f1=0.6521
Epoch 6: train_loss=0.6170 val_loss=0.6076 val_macro_f1=0.6714
Epoch 7: train_loss=0.5998 val_loss=0.6074 val_macro_f1=0.6730
Epoch 8: train_loss=0.5811 val_loss=0.6557 val_macro_f1=0.6043
Epoch 9: train_loss=0.5798 val_loss=0.5567 val_macro_f1=0.7027
Epoch 10: train_loss=0.5745 val_loss=0.6017 val_macro_f1=0.6486
Train summary:
{
  "accuracy": 0.7833660168416196,
  "balanced_accuracy": 0.7456819576725695,
  "macro_f1": 0.7380115169024519,
  "weighted_f1": 0.7859893318747756,
  "qwk": 0.8551706274502551,
  "mae": 0.22840004614142345,
  "rmse": 0.506276216443639,
  "spearman": 0.8103361157803475,
  "auroc_ovr": 0.9414027570311042,
  "ece": 0.04644550271534237
}
Val summary:
{
 